# Model 2: Neural Networks (Gu, Kelly & Xiu 2020)
Feed-forward NNs for predicting abnormal returns with congressional trading features.

In [1]:
import pandas as pd
import numpy as np
import json, os, warnings
warnings.filterwarnings('ignore')
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats
import plotly.graph_objects as go

SEED = 42
np.random.seed(SEED)
print('='*60)
print('MODEL 2: NEURAL NETWORKS')
print('='*60)

MODEL 2: NEURAL NETWORKS


In [2]:
# CONFIG
INPUT_PATH = 'data/prediction_bases/panel_final_stock_month.parquet'
FEATURES_PATH = 'data/prediction_bases/feature_sets.json'
OUTPUT_DIR = 'Model_2'
os.makedirs(OUTPUT_DIR, exist_ok=True)
TARGET = 'ret_future_car30_ff3'
MIN_TRAIN_MONTHS = 36
print(f'Target: {TARGET}')

Target: ret_future_car30_ff3


## 1. Load Data

In [3]:
print('\n[1] LOADING DATA...')
panel = pd.read_parquet(INPUT_PATH)
with open(FEATURES_PATH, 'r') as f:
    feature_sets = json.load(f)
print(f'    Panel: {panel.shape}')
print(f'    Congress: {len(feature_sets["congress"])} | Market: {len(feature_sets["market"])}')


[1] LOADING DATA...
    Panel: (43381, 119)
    Congress: 71 | Market: 29


In [4]:
df = panel.dropna(subset=[TARGET]).copy()
if 'month_dt' not in df.columns:
    df['month_dt'] = pd.to_datetime(df['month'].astype(str))
df = df.sort_values('month_dt').reset_index(drop=True)
print(f'    Observations: {len(df):,}')

    Observations: 34,623


## 2. Feature Sets

In [5]:
FEATURES_MARKET = [c for c in feature_sets['market'] if c in df.columns and 'car_' not in c.lower()]
FEATURES_CONGRESS = [c for c in feature_sets['congress'] if c in df.columns]
FEATURES_ALL = FEATURES_MARKET + FEATURES_CONGRESS
for col in FEATURES_ALL:
    if df[col].isnull().any(): df[col] = df[col].fillna(df[col].median())
print(f'Market: {len(FEATURES_MARKET)} | Congress: {len(FEATURES_CONGRESS)} | All: {len(FEATURES_ALL)}')

Market: 29 | Congress: 71 | All: 100


## 3. NN Architectures

In [6]:
NN_ARCHITECTURES = {
    'NN1': (32,),
    'NN2': (32, 16),
    'NN3': (32, 16, 8),
    'NN4': (64, 32, 16)
}
for k, v in NN_ARCHITECTURES.items(): print(f'  {k}: {v}')

  NN1: (32,)
  NN2: (32, 16)
  NN3: (32, 16, 8)
  NN4: (64, 32, 16)


## 4. Hyperparameters

In [7]:
HYPERPARAMS = {
    'activation': 'relu', 'solver': 'adam', 'alpha': 0.001,
    'learning_rate': 'adaptive', 'learning_rate_init': 0.001,
    'max_iter': 300, 'early_stopping': True, 'validation_fraction': 0.15,
    'n_iter_no_change': 15, 'batch_size': 256, 'random_state': SEED
}
print('Hyperparameters set')

Hyperparameters set


## 5. Training Function

In [8]:
months = np.sort(df['month_dt'].unique())
start_test_idx = MIN_TRAIN_MONTHS
print(f'Total months: {len(months)} | Test periods: {len(months) - start_test_idx}')

Total months: 149 | Test periods: 113


In [9]:
def train_nn(df, features, target, months, hidden_sizes, name, spec):
    all_preds, all_actuals, all_months = [], [], []
    history = []
    scaler = StandardScaler()
    test_indices = list(range(start_test_idx, len(months)))
    
    for i, t in enumerate(test_indices):
        train_m = months[:t]
        test_m = months[t]
        train_idx = df[df['month_dt'].isin(train_m)].index
        test_idx = df[df['month_dt'] == test_m].index
        if len(train_idx) < 100 or len(test_idx) == 0: continue
        
        X_tr = scaler.fit_transform(df.loc[train_idx, features])
        X_te = scaler.transform(df.loc[test_idx, features])
        y_tr = df.loc[train_idx, target].values
        y_te = df.loc[test_idx, target].values
        
        model = MLPRegressor(hidden_layer_sizes=hidden_sizes, **HYPERPARAMS, verbose=False)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        
        all_preds.extend(preds)
        all_actuals.extend(y_te)
        all_months.extend([test_m] * len(y_te))
        history.append({'month': test_m, 'n_iter': model.n_iter_})
        
        if (i+1) % 30 == 0: print(f'    {spec}-{name}: {i+1}/{len(test_indices)}')
    
    return {'preds': np.array(all_preds), 'actuals': np.array(all_actuals), 'months': all_months, 'history': pd.DataFrame(history)}

## 6. Train Models

In [ ]:
print('\n[6] TRAINING (10-20 min)...')
results = {}
for nn_name, hidden in NN_ARCHITECTURES.items():
    print(f'\n  {nn_name} {hidden}')
    results[f'{nn_name}_Base'] = train_nn(df, FEATURES_MARKET, TARGET, months, hidden, nn_name, 'Base')
    results[f'{nn_name}_Aug'] = train_nn(df, FEATURES_ALL, TARGET, months, hidden, nn_name, 'Aug')
print('\nDone!')


[6] TRAINING (10-20 min)...

  NN1 (32,)


## 7. Metrics

In [ ]:
def r2_oos(y, p):
    m = ~(np.isnan(y)|np.isnan(p))
    yt, yp = y[m], p[m]
    if len(yt)==0: return np.nan
    return 1 - np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2)

metrics = []
for k, r in results.items():
    parts = k.rsplit('_', 1)
    r2 = r2_oos(r['actuals'], r['preds'])
    metrics.append({'Model': parts[0], 'Spec': parts[1], 'R2_pct': r2*100, 'N': len(r['actuals'])})

metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

In [ ]:
pivot = metrics_df.pivot(index='Model', columns='Spec', values='R2_pct')
pivot['Impr_pp'] = pivot['Aug'] - pivot['Base']
pivot = pivot.sort_values('Aug', ascending=False)
print('\nR² Comparison:')
print(pivot.round(4))

## 8. Clark-West Test

In [ ]:
def cw_test(y, f1, f2):
    m = ~(np.isnan(y)|np.isnan(f1)|np.isnan(f2))
    y,f1,f2 = y[m],f1[m],f2[m]
    if len(y)<30: return np.nan, np.nan
    e1, e2 = y-f1, y-f2
    d = e1**2 - (e2**2 - (f1-f2)**2)
    if d.std()==0: return np.nan, np.nan
    cw = d.mean()/(d.std(ddof=1)/np.sqrt(len(d)))
    return cw, 1-stats.norm.cdf(cw)

cw_res = []
for nn in NN_ARCHITECTURES.keys():
    act = results[f'{nn}_Base']['actuals']
    pb = results[f'{nn}_Base']['preds']
    pa = results[f'{nn}_Aug']['preds']
    cw, pv = cw_test(act, pb, pa)
    cw_res.append({'Model': nn, 'CW': round(cw,3) if not np.isnan(cw) else np.nan, 'p': round(pv,4) if not np.isnan(pv) else np.nan})

cw_df = pd.DataFrame(cw_res)
print(cw_df.to_string(index=False))

## 9. Compare with Model 1

In [ ]:
try:
    m1 = pd.read_csv('Model_1/r2_comparison.csv', index_col=0)
    print('Model 1 R²:')
    print(m1[['Base','Augmented','Improvement_pp']].round(4))
    m1_ok = True
except:
    print('Model 1 not found')
    m1_ok = False

## 10. Visualizations

In [ ]:
# Fig 1: R² Comparison
fig = go.Figure()
fig.add_trace(go.Bar(name='Base', x=list(pivot.index), y=pivot['Base'], marker_color='#64748B'))
fig.add_trace(go.Bar(name='Augmented', x=list(pivot.index), y=pivot['Aug'], marker_color='#8B5CF6'))
fig.update_layout(title='Neural Network R² OOS', barmode='group', yaxis_title='R² (%)',
                  plot_bgcolor='#F8FAFC', width=700, height=400)
fig.show(renderer='browser')

In [ ]:
# Fig 2: Improvement
impr = pivot['Impr_pp'].sort_values()
colors = ['#10B981' if x>0 else '#EF4444' for x in impr]
fig = go.Figure(go.Bar(y=impr.index, x=impr.values, orientation='h', marker_color=colors))
fig.add_vline(x=0, line_color='#374151')
fig.update_layout(title='Improvement from Congress Features', xaxis_title='pp',
                  plot_bgcolor='#F8FAFC', width=650, height=380)
fig.show(renderer='browser')

In [ ]:
# Fig 3: Training iterations
best_nn = pivot['Aug'].idxmax()
hist = results[f'{best_nn}_Aug']['history']
fig = go.Figure(go.Histogram(x=hist['n_iter'], marker_color='#8B5CF6', nbinsx=25))
fig.update_layout(title=f'Training Iterations ({best_nn})', xaxis_title='Iterations',
                  plot_bgcolor='#F8FAFC', width=650, height=350)
fig.show(renderer='browser')

In [ ]:
# Fig 4: Pred vs Actual
act = results[f'{best_nn}_Aug']['actuals']
prd = results[f'{best_nn}_Aug']['preds']
idx = np.random.choice(len(act), min(4000, len(act)), replace=False)
fig = go.Figure()
fig.add_trace(go.Scattergl(x=act[idx], y=prd[idx], mode='markers', marker=dict(size=3, opacity=0.3, color='#8B5CF6')))
fig.add_trace(go.Scatter(x=[act.min(),act.max()], y=[act.min(),act.max()], mode='lines', line=dict(dash='dash', color='red')))
fig.update_layout(title=f'Predicted vs Actual ({best_nn})', xaxis_title='Actual', yaxis_title='Predicted',
                  plot_bgcolor='#F8FAFC', width=550, height=500)
fig.show(renderer='browser')

## 11. Export

In [ ]:
metrics_df.to_csv(f'{OUTPUT_DIR}/nn_metrics.csv', index=False)
pivot.to_csv(f'{OUTPUT_DIR}/nn_r2_comparison.csv')
cw_df.to_csv(f'{OUTPUT_DIR}/nn_clark_west.csv', index=False)
with open(f'{OUTPUT_DIR}/hyperparams.json', 'w') as f: json.dump(HYPERPARAMS, f, indent=2)
print(f'Saved to {OUTPUT_DIR}/')

## 12. Summary

In [ ]:
print('='*60)
print('SUMMARY')
print('='*60)
best = pivot['Aug'].idxmax()
print(f'Best NN: {best}')
print(f'  R² Base: {pivot.loc[best,"Base"]:.4f}%')
print(f'  R² Aug:  {pivot.loc[best,"Aug"]:.4f}%')
print(f'  Improvement: {pivot.loc[best,"Impr_pp"]:+.4f} pp')
print('\nClark-West p-values:')
for _, r in cw_df.iterrows():
    sig = '**' if r['p']<0.05 else '*' if r['p']<0.10 else ''
    print(f'  {r["Model"]}: p={r["p"]} {sig}')
print('='*60)